### Text Spinning
- Es la tecnica de tomar un articulo ya escrito y modificarlo (reemplazando, palabras, frases, reorganizando estructuras) para que parezca un nuevo contenido, pero manteniendo el mensaje original

#### importancia de los motores de busqueda
- factores que influencian el ranking
- el desafio de crear contenido original y abundante
- tiempo y esfuerzo requeridos para escribir
#### Evolución de las tecnicas de generación de contenido
- uso de software de sugerencia de palabras
- limitaciones de las tecnicas previas a machine learning
- los modelos de markov como técnica básica
#### avances tecnicos en NLP
- mejoras por sobre los modelos de markov
- técnicas avanzadas como RNNs y transformers

### N-GRAM modelos de gran escala


In [ ]:
import numpy as np
import pandas as pd 
from nltk import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer 

: 

In [ ]:
# descargar el conjunto de datos del tokenizador para español
import nltk
nltk.download('punkt')

: 

In [ ]:
# carga de datos
df = pd.read_csv(r'C:\Users\ander\Escritorio\PYTHON\Python 2025\Procesamiento_lenguaje_natural_NLP\datasets\df_total.csv', sep=',', encoding='utf-8')
df.head(5)


In [ ]:
textos = df['news']
textos[0]

In [ ]:
textos.shape

In [ ]:
# diccionario
# key: (w(t-I), W(t+I)), value:{w(t): count(w(t))}
prob = {}
# recorremos todos los textos
for doc in textos:
    lineas = doc.split('.')
    # recorremos cada una de las lineas
    for linea in lineas:
        tokens = word_tokenize(linea, language='spanish')
        if  len(tokens) >= 2:
            for i in range(len(tokens) -2):
                t_0 = tokens[i]
                t_1 = tokens[i + 1]
                t_2 = tokens[i + 2]
                key = (t_0, t_2)
                if key not in prob:
                    prob[key] = {} 
                if t_1 not in prob[key]:
                    prob[key][t_1] = 1
                else:
                    prob[key][t_1] += 1
                    

In [ ]:
prob 

In [ ]:
# normalizar las probabilidades
for key, d in prob.items():
    total = sum(d.values())
    for k, v in d.items():
        d[k] = v / total
prob

In [ ]:
# detokenizar
detokenizar = TreebankWordDetokenizer()
ejemplo = 'Hola, ¿cómo estás?'
token_ej = word_tokenize(ejemplo, language = 'spanish')
print(detokenizar.detokenize(token_ej))

In [ ]:
# funcion palabra random
def sample_word(d):
    p0 = np.random.random()
    cumulative = 0
    for t, p in d.items():
        cumulative += p
        if p0 < cumulative:
            return t 

In [ ]:
# funcion para spinnear una linea
def spin_line(linea):
    tokens = word_tokenize(linea, language='spanish') 
    i = 0
    salida = [tokens[0]] 
    if len(tokens) >= 2:
        while i < (len(tokens) -2):
            t_0 = tokens[i]
            t_1 = tokens[i + 1]
            t_2 = tokens[i + 2]
            key = (t_0, t_2)
            p_dist = prob[key]
            if len(p_dist) > 1 and np.random.random() < 0.3:
                middle = sample_word(p_dist)
                salida.append(t_1)
                salida.append('<' + middle + '>')  
                salida.append(t_2)
                i += 2
            else:
                salida.append(t_1)
                i += 2
        if i == len(tokens) -2:
            salida.append(tokens[-1])
    return detokenizar.detokenize(salida)

In [ ]:
# funcion para hacer el spinner
def spin_document(doc):
    lineas = doc.split('.')
    output = []
    for linea in lineas:
        if linea:
            new_line = spin_line(linea)
        else:
            new_line = linea
        output.append(new_line)
    return '\n'.join(output)
            

In [ ]:
print(spin_line('el desarrollo sostenible es importante'))


In [ ]:
# texto random del documento 
i = np.random.choice(textos.shape[0])
doc = textos.iloc[i] 
new_doc = spin_document(doc)
new_doc